# 04. Behaviour cloning

Walks through the BC toolchain end-to-end:

1. **Inspect** the shipped BC teacher dataset.
2. **Train** a BC agent for 3 epochs (smoke budget).
3. **Evaluate** the BC agent.
4. **(Reference)** Show how to generate fresh BC data and how to chain
   into PPO.

Smoke budget end to end: ~3 min on CPU. The thesis BC recipe (UECD-BC:
30 epochs on the full dataset, ~30 min on GPU) is in
[`experiments/BC/train_UECD-BC.slurm`](../experiments/BC/train_UECD-BC.slurm).

Prereq: [`00_navigate.ipynb`](00_navigate.ipynb) must be green.

## 1. The shipped BC dataset

[`data/BC/training/`](../data/BC/training/) ships 6 NPZ chunks:
RAISocketAI plays 100 games against each of three opponents (itself,
CoacAI, Mayari) on `basesWorkers16x16A`. Each transition records the
demonstrator's per-cell observation (74 channels) + the gridnet action
it emitted + the sparse reward at that step.

In [ ]:
import glob

import numpy as np

from microrts_agent.paths import PROJECT_ROOT

bc_dir = PROJECT_ROOT / "data" / "BC" / "training"
chunks = sorted(glob.glob(str(bc_dir / "bc_chunk_*.npz")))
print(f"Chunks: {len(chunks)}\n")

total = 0
for cp in chunks:
    d = np.load(cp)
    n = len(d["obs"])
    total += n
    print(f"  {cp.split('/')[-1]:38s} obs={d['obs'].shape} actions={d['actions'].shape} n={n:,}")
print(f"\nTotal transitions: {total:,}")

## 2. Train a BC agent (3 epochs, smoke)

`microrts-agent bc train` runs supervised training: cross-entropy on the
demonstrator's actions, MSE on a value head (optional). Output dir gets
an `agent.pt` + `config.json` (same format as the PPO trainer).

In [ ]:
import subprocess

bc_run_dir = PROJECT_ROOT / "outputs" / "runs" / "notebook-bc_s1"

cmd = [
    "microrts-agent",
    "bc",
    "train",
    "--data",
    *chunks,
    "--architecture",
    "gridnet",
    "--epochs",
    "3",
    "--batch-size",
    "128",
    "--lr",
    "1e-4",
    "--seed",
    "1",
    "--output",
    str(bc_run_dir),
]
print(f"Training command (--data summarised, {len(chunks)} chunks):")
print(
    "  " + " ".join(cmd[:5]) + f" <{len(chunks)} chunks>" + " " + " ".join(cmd[5 + len(chunks) :])
)
result = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True, timeout=900)
print("\n" + result.stdout[-1200:])
if result.returncode != 0:
    print("--- stderr ---")
    print(result.stderr[-1200:])

## 3. Evaluate the BC agent

3 epochs is severely under-trained: this is a pipeline smoke test, not
a benchmark. The published BC-only number (UECD-BC, 30 epochs) is **78
% pool WR** -- see [`data/BC/baseline/`](../data/BC/baseline/) for the
per-opponent breakdown.

In [ ]:
result = subprocess.run(
    [
        "microrts-agent",
        "evaluate",
        "--agent",
        str(bc_run_dir),
        "--opponent",
        "RandomBiasedAI",
        "--maps",
        "maps/open_competition/basesWorkers16x16A.xml",
        "--nb_games",
        "3",
        "--max-steps",
        "2000",
    ],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    timeout=300,
)
print(result.stdout[-1200:])

## 4. (Reference) Generate fresh BC data

`microrts-agent bc generate` records games between a demonstrator bot and a list of opponents, writing one NPZ chunk per opponent per game-batch under `outputs/bc_data/`. The shipped 6 chunks under `data/BC/training/` were produced by exactly this command (driver: [`experiments/BC/generate_BC_dataset.slurm`](../experiments/BC/generate_BC_dataset.slurm)).

Don't actually run this in the notebook (would take 30+ min CPU); the snippet below is the canonical invocation:

```python
subprocess.run([
    "microrts-agent", "bc", "generate",
    "--bot", "RAISocketAI",
    "--opponents", "RAISocketAI", "CoacAI", "Mayari",
    "--games-per-opponent", "100",
    "--map", "maps/open_competition/basesWorkers16x16A.xml",
])
```

## 5. (Reference) Chain BC into PPO

The dissertation's UECD-BC-PPO agent takes a trained BC agent and resumes PPO training from it, optionally with a KL teacher penalty against the frozen BC policy. That lifts the BC-only 78 % baseline to ~96 % pool WR after 100 M PPO steps.

The CLI uses `--load-model` on `microrts-agent train`:

```python
subprocess.run([
    "microrts-agent", "train",
    "--exp-name", "my-bc-ppo_s1",
    "--architecture", "gridnet",
    "--total-timesteps", "100000000",
    "--load-model", str(bc_run_dir / "agent.pt"),  # warm-start from BC
    # ...
])
```

The thesis-grade driver is [`experiments/BC/train_UECD-BC-PPO.slurm`](../experiments/BC/train_UECD-BC-PPO.slurm).

## Next steps

- Browse the BC baseline numbers: [`data/BC/baseline/results.csv`](../data/BC/baseline/results.csv) (per-opponent breakdown, 1000-game eval per cell).
- Full reproduction: [`experiments/BC/train_UECD-BC.slurm`](../experiments/BC/train_UECD-BC.slurm) on a GPU node.
- The BC + PPO scaling curve is plotted in [`dissertation/figs/figs-python/bc_vs_scratch_overall.py`](../dissertation/figs/figs-python/bc_vs_scratch_overall.py) (compares against from-scratch training of the same architecture).